# PKG edge tables: build and monthly refresh

**PNC Treasury Management · DSI · Payment Knowledge Graph**

This notebook builds and refreshes the deduplicated edge tables behind the PKG customer and counterparty metrics, from
the Neo4j staging table `dsihd01p_dsi.neo4j_payments`. It replaces `pkg_edges_2024_backfill.ipynb`: the same notebook
does the full backfill and the monthly refresh. Output goes to `bdahd01p_dlcdi1_cdi_tm` as external parquet tables
partitioned by `month` ('YYYY-MM').

**How to run.** Run all cells. A month is built when it is missing, was built with other settings, or its source changed
since it was built. Every other month is skipped, so an interrupted run continues where it stopped. Don't run two copies
at the same time.

**What happens to each month**
1. The month's source rows are read once and cached.
2. **Completeness gate.** Rows, dollars and the last date of each rail are compared with the previous months. A month
   that looks incomplete stops the run before anything is written.
3. The 5C/6C duplicate pairs are found, reading one extra day before the month.
4. The outputs are written to a hidden staging folder (`_staging`).
5. **Exact reconciliation.** Source rows, daily edges, monthly edges, counterparty-month and the attribute table must
   agree on rows and amounts in every category. Any mismatch stops the run, and nothing is published.
6. **Publish.** Each staged folder is renamed into place with a build marker (`_PKG_BUILD`), and the month is logged in
   `pkg_edge_runlog`. Readers see the previous version of the month until the switch.

Each run also re-checks the source of the last `RECHECK_LAST` built months and rebuilds any whose source changed
(late or restated data).

**Cost.** The previous notebook took about 10 minutes and 7 GB per month with the same logic. The full
2024-01 → 2026-09 build is about 6 hours; a monthly refresh is one month plus the re-check.

## Output tables

| Table | Grain | Columns |
|---|---|---|
| `pkg_edge_daily` | one row per (txn_date, src, dst, category) | txn_date, src_id, src_kind, dst_id, dst_kind, category, cpty_id_src, is_self, amount, n_txn, n_non_usd, **amt_min, amt_max** |
| `pkg_edge_monthly` | one row per (src, dst, category) per month | the same keys without txn_date; amount, n_txn, n_non_usd, active_days, first_dt, last_dt, **amt_min, amt_max** |
| `pkg_dedup_5c6c` | one row per dropped 6C duplicate | trans_id_drop, trans_id_keep, tier, dt_drop, dt_keep, acct, amount, cpty_id_drop, cpty_id_keep |
| `pkg_cpty_month` | counterparty × category × type × name × FI per month | cpty_id, cpty_id_src, category, cpty_type, cpty_name, fi, n_txn, amount |
| `pkg_cpty_attr_month` **(new)** | counterparty × category × attribute values per month, only for the categories in `ATTR_FIELDS` | cpty_id, cpty_id_src, category, merchant_id, merchant_city, merchant_state, merchant_zip, merchant_cat_cd, originating_company, n_txn, amount |
| `pkg_edge_runlog` **(new)** | one row per month attempt | build_id, month, status (published / stopped / failed), reason, gate result, source fingerprint, totals, minutes |
| `pkg_edge_src_profile` **(new)** | source profile per month × category | n_rows, amount, non-USD rows, active days, last date, latest load time |

**Unchanged from the previous notebook:** every column that existed before is computed exactly as before.
**No longer built here:** `pkg_cpty_dim` and the v1 entity map (old §5) move to the reference-data notebook, and the
one-time sensitivity check (old §6) is dropped.

**Column semantics.**
- `src_kind` / `dst_kind`: `CUST` = a PNC customer, identified by `mdm_id`; `CPTY` = an external counterparty.
- `amount`: the sum of the absolute transaction amounts, DECIMAL(18,2).
- `amt_min` / `amt_max`: the smallest and largest single payment in the edge. They are equal when every payment had the
  same amount, which lets the periodicity metrics recognise fixed-amount payments even when a pair pays several times a day.
- `n_non_usd`: rows whose `trans_currency` isn't USD. Their amounts are summed as-is, pending confirmation that
  `trans_amt` is USD-converted.

## Rules and why

| Rule | Reason |
|---|---|
| Debit card (8B/8C) excluded | about 45% of rows but 0.5% of dollars; outside the B2B scope |
| 5C/6C deduplication: drop the 6C twin, keep 5C | 6C (P2P feed) re-records about 21% of its rows from 5C (PRT feed): same account, amount, date and name. 5C carries the counterparty bank; 6C never does. Tier 1 = same day, tier 2 = 6C dated one day later (follows daylight saving time) |
| Counterparty id = `unq_cpty_id` for 3B, 3C, 7B, 9B; `unq_cpty_acct_id` for all other categories | for 3B/3C, `unq_cpty_acct_id` is only the originator's 8-digit routing number; 7B and 9B have no account id. Elsewhere the account id is stable across name spellings. `cpty_id_src` = A / B records which was used |
| Customers keyed by current `mdm_id` (not `mdm_id_*_orig`) | a stable entity after MDM merges |
| Self-transfers kept, flagged `is_self` | same `mdm_id` on both sides; up to 34% of internal wires |
| Counterparty attributes in their own table | merchant fields (PCARD) give counterparty locations and a merchant category code; `originating_company` (3B/3C) names the ACH originator. A separate table keeps `pkg_cpty_month` exactly as before, so nothing downstream shifts |
| `trans_purpose` not carried | waits on the PII review |

## For downstream notebooks

A month is ready when `pkg_edge_runlog` has a `published` row for it. The latest build of each month (Spark SQL):

```sql
SELECT month, MAX_BY(build_id, finished_at) AS build_id, MAX(finished_at) AS built_at
FROM bdahd01p_dlcdi1_cdi_tm.pkg_edge_runlog WHERE status = 'published' GROUP BY month
```

## Open questions with the data owners
- How `unq_cpty_id` is constructed; what `np_key` and `mdm_id_*_orig` mean.
- Whether `trans_amt` is USD-converted for non-USD rows.
- How returns and reversals are represented (treated as not occurring for now).
- The source history retention: once old months leave the source, these tables are the only copy.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
SRC_TABLE = "dsihd01p_dsi.neo4j_payments"
TGT_DB    = "bdahd01p_dlcdi1_cdi_tm"
PREFIX    = "pkg_"

# Which months
RUN_FROM = "2024-01"            # first month of the edge history
RUN_TO   = None                 # None = last complete calendar month
RUN_ONLY = None                 # e.g. ["2026-07", "2026-08"] for a trial; overrides RUN_FROM / RUN_TO
FORCE    = []                   # months to rebuild even if built and unchanged
RECHECK_LAST       = 2          # re-check the source of the last N built months; rebuild if it changed
MAX_MONTHS_PER_RUN = None       # stop after building N months (split a backfill over sessions)

# Completeness gate (§2). Rails: ACH = 1–3, WIRE = 4, RTP = 5–6, CHECK = 7, PCARD = 9
GATE_WINDOW    = 3              # compare with the median of the previous N months
GATE_ROWS_LOW  = 0.25           # STOP when a rail has more than 25% fewer rows than that median
GATE_ROWS_HIGH = 0.25           # warn when it has more than 25% more
GATE_USD       = 0.50           # warn when a rail's dollars move more than 50% either way
GATE_TAIL_DAYS = {"ACH": 4, "WIRE": 4, "RTP": 2, "CHECK": 4, "PCARD": 4}   # STOP when a rail's last date is more
                                                                            # than N days before month end
ACCEPT_MONTHS  = []             # months allowed through a STOP after you checked them

# Edge rules. These are part of the build signature: changing one rebuilds every month.
EXCLUDE_CODES  = {"8B", "8C"}                                   # debit card
B_ID_CODES     = {"3B", "3C", "7B", "8B", "8C", "9B"}           # counterparty id = unq_cpty_id here
KEEP_CAT, DROP_CAT = "5C.RTP_PRT_CPTY_PAYS", "6C.RTP_P2P_CPTY_PAYS"
EXPECTED_CODES = {"1A1", "1A2", "1B", "1C", "2A", "2B", "2C", "3B", "3C", "4A", "4B", "4C", "5A", "5B", "5C",
                  "6A", "6B", "6C", "7A", "7B", "7C", "8B", "8C", "9B"}   # a new code stops the run
RAIL_OF_DIGIT  = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP", "6": "RTP",
                  "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}

# Counterparty attributes for pkg_cpty_attr_month: output column -> (source column, categories it is kept for)
ATTR_FIELDS = {
    "merchant_id":         ("merchant_id",         ["9B"]),
    "merchant_city":       ("merchant_city",       ["9B"]),
    "merchant_state":      ("merchant_state",      ["9B"]),
    "merchant_zip":        ("merchant_zip",        ["9B"]),
    "merchant_cat_cd":     ("merchant_cat_cd",     ["9B"]),
    "originating_company": ("originating_company", ["3B", "3C"]),
}
PROFILE_ATTRS = True            # §5: coverage of these fields in every category (latest month built)
LOAD_TS_COL   = "hdfs_load_ts"  # part of the source fingerprint; set None if the column doesn't exist

EDGE_VERSION = "edges_v2"       # bump when the build logic changes (forces a rebuild)
FILES = {"dedup_5c6c": 1, "edge_daily": 8, "edge_monthly": 4, "cpty_month": 4, "cpty_attr_month": 1}
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400
SHOW_SQL = False                # §0b prints the SQL of every step for one month

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, datetime as dt, hashlib, html as _html, json, time, uuid
from contextlib import contextmanager
from decimal import Decimal
from urllib.parse import unquote
from IPython.display import display, HTML
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("pkg_edges_refresh")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")              # keep Spark warnings out of the cell outputs
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- months
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
def month_bounds(m):
    y, mo = map(int, m.split("-"))
    return f"{m}-01", f"{m}-{calendar.monthrange(y, mo)[1]:02d}"
def prev_months(m, n):
    y, mo = map(int, m.split("-")); out = []
    for _ in range(n):
        y, mo = (y - 1, 12) if mo == 1 else (y, mo - 1); out.insert(0, f"{y:04d}-{mo:02d}")
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"
now_iso = lambda: dt.datetime.now().isoformat(timespec="seconds")
code_of = lambda c: c.split(".")[0]
rail_of = lambda c: RAIL_OF_DIGIT.get(code_of(c)[:1], "OTHER")

# ---- one consolidated HTML output per cell
def _to_num(pdf):
    pdf = pdf.copy()
    for c in pdf.columns:
        if pdf[c].dtype == object and pdf[c].map(lambda v: isinstance(v, Decimal)).any():
            pdf[c] = pdf[c].map(lambda v: float(v) if isinstance(v, Decimal) else v).astype(float)
    return pdf
def tbl(pdf, pct=(), dec=(), flag=None):
    pdf = _to_num(pdf)
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    if flag and flag in pdf.columns:
        colors = {"STOP": "#f8d0d0", "FAILED": "#f8d0d0", "WARN": "#fbe7c0", "ACCEPTED": "#fbe7c0",
                  "OK": "#d5efd8", "True": "#d5efd8", "False": "#f8d0d0"}
        _f = lambda v: f"background-color: {colors.get(str(v), '')}"
        sty = sty.map(_f, subset=[flag]) if hasattr(sty, "map") else sty.applymap(_f, subset=[flag])
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(parts) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def mkdirs(p):
    fs_for(p).mkdirs(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    assert not exists(b), f"rename target already exists: {b}"
    if not fs_for(a).rename(JPath(a), JPath(b)):
        raise RuntimeError(f"rename failed: {a} -> {b}")
def list_names(p, dirs_only=False):
    if not exists(p): return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if (s.isDirectory() or not dirs_only)]
def write_text(p, text):
    o = fs_for(p).create(JPath(p), True); o.write(bytearray(text.encode("utf-8"))); o.close()
def read_text(p):
    if not exists(p): return None
    jio = spark._jvm.java.io
    r = jio.BufferedReader(jio.InputStreamReader(fs_for(p).open(JPath(p)), "UTF-8")); lines = []
    line = r.readLine()
    while line is not None:
        lines.append(line); line = r.readLine()
    r.close()
    return "\n".join(lines)
def size_of(p):
    if not exists(p): return 0, 0
    c = fs_for(p).getContentSummary(JPath(p)); return c.getLength(), c.getFileCount()
GB = 1e9

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    TIMINGS.append({"month": month, "stage": stage, "seconds": time.time() - t0})

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__} | last complete month {LAST_COMPLETE}")

## §0 Source and target checks

- **Source.** Every category code must be known (`EXPECTED_CODES`), and every column the build uses must exist.
- **Target.** Tables are created when missing. New columns at the end (`amt_min` / `amt_max` on the edge tables) are
  added in place; any other column change re-creates the table definition and re-attaches every month. Only external
  tables are touched, and data files are never deleted. Months built by the previous notebook read the new columns as
  empty until they are rebuilt, which this run does.

In [ ]:
# ---- §0 Source and target checks -----------------------------------------------------------------
_parts = [r[0] for r in spark.sql(f"SHOW PARTITIONS {SRC_TABLE}").collect()]          # metastore only, no data read
SRC_CATS, SRC_DATES = set(), set()
for _p in _parts:
    _kv = {k.lower(): unquote(v) for k, v in (x.split("=", 1) for x in _p.split("/"))}
    SRC_CATS.add(_kv["category"]); SRC_DATES.add(_kv["trans_dt"])
_unexpected = sorted(c for c in SRC_CATS if code_of(c) not in EXPECTED_CODES)
assert not _unexpected, (f"New category in the source: {_unexpected}. Decide how to treat it (add its code to "
                         "EXPECTED_CODES, and to EXCLUDE_CODES or B_ID_CODES when needed) before building.")
assert KEEP_CAT in SRC_CATS and DROP_CAT in SRC_CATS, "5C / 6C category names not found in the source"
KEEP_CATS = sorted(c for c in SRC_CATS if code_of(c) not in EXCLUDE_CODES)
SRC_FIRST_MONTH, SRC_LAST_DATE = min(SRC_DATES)[:7], max(SRC_DATES)

_src_cols = {c.lower() for c in spark.table(SRC_TABLE).columns}
_need = {"trans_id", "trans_dt", "trans_amt", "category", "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_receives",
         "unq_cpty_acct_id", "unq_cpty_id", "cpty_name", "cpty_type", "cpty_fin_entity_name", "trans_currency"}
assert not (_need - _src_cols), f"source is missing columns {sorted(_need - _src_cols)}"
_attr_missing = sorted({src for src, _ in ATTR_FIELDS.values()} - _src_cols)
assert not _attr_missing, f"ATTR_FIELDS source columns not found: {_attr_missing}. Fix the names or remove the fields."
assert LOAD_TS_COL is None or LOAD_TS_COL.lower() in _src_cols, f"{LOAD_TS_COL} not found: set LOAD_TS_COL = None"
ATTR_CODES = sorted({c for _, codes in ATTR_FIELDS.values() for c in codes})
assert all(c in EXPECTED_CODES and c not in EXCLUDE_CODES for c in ATTR_CODES), "ATTR_FIELDS lists an unknown or excluded category"

# Build signature: every setting that changes the content of a month
SIG = hashlib.md5(json.dumps({
    "version": EDGE_VERSION, "src": SRC_TABLE, "exclude": sorted(EXCLUDE_CODES), "b_id": sorted(B_ID_CODES),
    "keep": KEEP_CAT, "drop": DROP_CAT, "attr": {k: [v[0], sorted(v[1])] for k, v in sorted(ATTR_FIELDS.items())},
}, sort_keys=True).encode()).hexdigest()[:12]

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
REPL = int(fs_for(DB_LOC).getDefaultReplication(JPath(DB_LOC)))
KEYS = ["dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month"] + (["cpty_attr_month"] if ATTR_FIELDS else [])
LOG_KEYS = ["edge_runlog", "edge_src_profile"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS + LOG_KEYS}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS + LOG_KEYS}

_edge_keys = [("src_id", "string"), ("src_kind", "string"), ("dst_id", "string"), ("dst_kind", "string"),
              ("category", "string"), ("cpty_id_src", "string"), ("is_self", "boolean")]
SCHEMA = {
    "dedup_5c6c": [("trans_id_drop", "string"), ("trans_id_keep", "string"), ("tier", "int"), ("dt_drop", "date"),
                   ("dt_keep", "date"), ("acct", "string"), ("amount", "decimal(18,2)"),
                   ("cpty_id_drop", "string"), ("cpty_id_keep", "string")],
    "edge_daily": [("txn_date", "date")] + _edge_keys + [
                   ("amount", "decimal(18,2)"), ("n_txn", "int"), ("n_non_usd", "int"),
                   ("amt_min", "decimal(18,2)"), ("amt_max", "decimal(18,2)")],
    "edge_monthly": _edge_keys + [
                   ("amount", "decimal(18,2)"), ("n_txn", "int"), ("n_non_usd", "int"), ("active_days", "int"),
                   ("first_dt", "date"), ("last_dt", "date"), ("amt_min", "decimal(18,2)"), ("amt_max", "decimal(18,2)")],
    "cpty_month": [("cpty_id", "string"), ("cpty_id_src", "string"), ("category", "string"), ("cpty_type", "string"),
                   ("cpty_name", "string"), ("fi", "string"), ("n_txn", "bigint"), ("amount", "decimal(22,2)")],
    "cpty_attr_month": [("cpty_id", "string"), ("cpty_id_src", "string"), ("category", "string")]
                   + [(k, "string") for k in ATTR_FIELDS] + [("n_txn", "bigint"), ("amount", "decimal(22,2)")],
    "edge_runlog": [("build_id", "string"), ("month", "string"), ("status", "string"), ("reason", "string"),
                   ("gate", "string"), ("gate_notes", "string"), ("sig", "string"), ("fingerprint", "string"),
                   ("max_load_ts", "string"), ("src_rows", "bigint"), ("dropped_6c", "bigint"), ("edge_rows", "bigint"),
                   ("amount", "decimal(22,2)"), ("daily_edges", "bigint"), ("monthly_edges", "bigint"),
                   ("minutes", "double"), ("started_at", "string"), ("finished_at", "string")],
    "edge_src_profile": [("month", "string"), ("category", "string"), ("n_rows", "bigint"), ("amount", "decimal(22,2)"),
                   ("non_usd_rows", "bigint"), ("n_days", "int"), ("max_dt", "date"), ("max_load_ts", "string"),
                   ("profiled_at", "string"), ("profile_source", "string")],
}
PARTITIONED = {k: k in KEYS for k in SCHEMA}
ddl = lambda key: ", ".join(f"`{c}` {t}" for c, t in SCHEMA[key])

def table_exists(t):
    db, name = t.split(".")
    return len(spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").collect()) > 0

def ensure_table(key):
    """Create the table, or bring its definition in line with SCHEMA. External tables only; data files are never touched:
    new trailing columns are added in place; any other change re-creates the definition (purge switched off first)."""
    t, part = T[key], " PARTITIONED BY (month STRING)" if PARTITIONED[key] else ""
    create = (f"CREATE EXTERNAL TABLE {t} ({ddl(key)}){part} STORED AS PARQUET LOCATION '{P[key]}' "
              "TBLPROPERTIES ('external.table.purge' = 'false')")
    if table_exists(t):
        have = [(f.name.lower(), f.dataType.simpleString()) for f in spark.table(t).schema
                if not (PARTITIONED[key] and f.name.lower() == "month")]
        want = SCHEMA[key]
        if have == want:
            return "ok"
        kind = [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Type"]
        assert kind and kind[0].strip().upper() == "EXTERNAL", f"{t} is not an external table; not touching it"
        if want[:len(have)] == have:                               # only new columns at the end
            added = ", ".join(f"`{c}` {ty}" for c, ty in want[len(have):])
            spark.sql(f"ALTER TABLE {t} ADD COLUMNS ({added})")
            return f"columns added: {', '.join(c for c, _ in want[len(have):])}"
        spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('external.table.purge' = 'false')")   # DROP must keep the files
        spark.sql(f"DROP TABLE {t}"); spark.sql(create)
        if PARTITIONED[key]:
            spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
        return "columns changed: definition re-created, months re-attached"
    mkdirs(P[key]); spark.sql(create)
    if PARTITIONED[key]:
        spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
    return "created"

TABLE_ACTIONS = {k: ensure_table(k) for k in KEYS + LOG_KEYS}

_codes = sorted({code_of(c) for c in SRC_CATS})
out(sec("Source", tbl(pd.DataFrame([{
        "table": SRC_TABLE, "first_date": min(SRC_DATES), "last_date": SRC_LAST_DATE, "partitions": len(_parts),
        "categories_kept": len(KEEP_CATS), "excluded": ", ".join(sorted(c for c in _codes if c in EXCLUDE_CODES)),
        "codes": " ".join(_codes)}]))),
    sec("Target tables", tbl(pd.DataFrame([{"table": T[k], "action": v, "partitioned": PARTITIONED[k],
        "months_attached": len([n for n in list_names(P[k], True) if n.startswith("month=")]) if PARTITIONED[k] else None}
        for k, v in TABLE_ACTIONS.items()]))),
    para(f"Location {DB_LOC} (replication {REPL}). Build signature {SIG}. Attribute fields: "
         + (", ".join(f"{k} ({'/'.join(v[1])})" for k, v in ATTR_FIELDS.items()) or "none") + "."))

## §0b SQL

Every table is built by the Spark SQL below; Python only loops over months, runs the gate and moves folders.
Set `SHOW_SQL = True` to print each statement for one month.

In [ ]:
# ---- §0b SQL -------------------------------------------------------------------------------------
def L(xs):
    """SQL list of string literals."""
    return ", ".join("'" + str(x).replace("\\", "\\\\").replace("'", "\\'") + "'" for x in sorted(xs))
def nz(c):
    """String, trimmed, blank -> NULL."""
    return f"NULLIF(TRIM(CAST({c} AS STRING)), '')"
WS = r"'\\s+'"                                   # regex \s+ as a Spark SQL literal

def where_months(ms):
    rng = " OR ".join(f"trans_dt BETWEEN '{month_bounds(m)[0]}' AND '{month_bounds(m)[1]}'" for m in ms)
    return f"({rng}) AND category IN ({L(KEEP_CATS)})"

def projection(where):
    """Source rows -> typed, trimmed columns and the counterparty id. Used for the build, the profile and the re-check."""
    raw  = "".join(f",\n             {nz(src)} AS a_{name}" for name, (src, _) in ATTR_FIELDS.items())
    keep = "".join(f", a_{name}" for name in ATTR_FIELDS)
    load = f"CAST({LOAD_TS_COL} AS STRING)" if LOAD_TS_COL else "CAST(NULL AS STRING)"
    return f"""
    SELECT trans_id, txn_date, category, code, mp, mr,
           CASE WHEN use_b THEN COALESCE(id_b, id_a) ELSE COALESCE(id_a, id_b) END AS cid,
           CASE WHEN use_b AND id_b IS NOT NULL THEN 'B' WHEN use_b AND id_a IS NOT NULL THEN 'A'
                WHEN id_a IS NOT NULL THEN 'A' WHEN id_b IS NOT NULL THEN 'B' END AS cid_src,
           amt, CASE WHEN ccy IS NOT NULL AND ccy <> 'USD' THEN 1 ELSE 0 END AS non_usd,
           cpty_name, cpty_type, fi{keep}, load_ts
    FROM (
      SELECT {nz('trans_id')} AS trans_id, TO_DATE(trans_dt) AS txn_date,
             {nz('category')} AS category, SUBSTRING_INDEX({nz('category')}, '.', 1) AS code,
             SUBSTRING_INDEX({nz('category')}, '.', 1) IN ({L(B_ID_CODES)}) AS use_b,
             {nz('mdm_id_pays')} AS mp, {nz('mdm_id_receives')} AS mr,
             {nz('unq_cpty_acct_id')} AS id_a, {nz('unq_cpty_id')} AS id_b,
             ABS(CAST(trans_amt AS DECIMAL(18,2))) AS amt, UPPER({nz('trans_currency')}) AS ccy,
             {nz('cpty_name')} AS cpty_name, {nz('cpty_type')} AS cpty_type, {nz('cpty_fin_entity_name')} AS fi{raw},
             {load} AS load_ts
      FROM {SRC_TABLE}
      WHERE {where}
    ) s"""

def profile_sql(rel):
    return f"""
    SELECT DATE_FORMAT(txn_date, 'yyyy-MM') AS month, category, COUNT(*) AS n_rows,
           CAST(SUM(amt) AS DECIMAL(22,2)) AS amount, CAST(SUM(non_usd) AS BIGINT) AS non_usd_rows,
           CAST(COUNT(DISTINCT txn_date) AS INT) AS n_days, MAX(txn_date) AS max_dt, MAX(load_ts) AS max_load_ts
    FROM {rel}
    GROUP BY DATE_FORMAT(txn_date, 'yyyy-MM'), category"""

def dedup_sql(m):
    """5C/6C pairs whose 6C row falls in month m. Reads one extra day so tier-2 pairs across the month start are found;
    pairing is one-to-one: the k-th 5C row of a (date, account, amount, name) group pairs with the k-th 6C row."""
    lo, hi = month_bounds(m)
    lo_ext = (dt.date.fromisoformat(lo) - dt.timedelta(days=1)).isoformat()
    grp = "PARTITION BY dt, acct, amt, nm ORDER BY trans_id"
    on = "x.dt = y.dt AND x.acct = y.acct AND x.amt = y.amt AND x.nm = y.nm AND x.rn = y.rn"
    return f"""
    WITH r AS (
      SELECT * FROM (
        SELECT {nz('trans_id')} AS trans_id, TO_DATE(trans_dt) AS dt, {nz('category')} AS category,
               {nz('pnc_dep_acct_receives')} AS acct, ABS(CAST(trans_amt AS DECIMAL(18,2))) AS amt,
               UPPER(REGEXP_REPLACE({nz('cpty_name')}, {WS}, ' ')) AS nm, {nz('unq_cpty_acct_id')} AS cid
        FROM {SRC_TABLE}
        WHERE trans_dt BETWEEN '{lo_ext}' AND '{hi}' AND category IN ({L([KEEP_CAT, DROP_CAT])})
      ) q WHERE acct IS NOT NULL AND nm IS NOT NULL
    ),
    a  AS (SELECT trans_id, dt, dt AS dt0, acct, amt, nm, cid, ROW_NUMBER() OVER ({grp}) AS rn FROM r WHERE category = '{KEEP_CAT}'),
    b  AS (SELECT trans_id, dt, dt AS dt0, acct, amt, nm, cid, ROW_NUMBER() OVER ({grp}) AS rn FROM r WHERE category = '{DROP_CAT}'),
    t1 AS (SELECT x.trans_id AS id_keep, y.trans_id AS id_drop, x.dt0 AS dt_keep, y.dt0 AS dt_drop, x.acct, x.amt,
                  x.cid AS cid_keep, y.cid AS cid_drop, 1 AS tier
           FROM a x JOIN b y ON {on}),
    a_left AS (SELECT r.* FROM (SELECT * FROM r WHERE category = '{KEEP_CAT}') r LEFT ANTI JOIN t1 ON r.trans_id = t1.id_keep),
    b_left AS (SELECT r.* FROM (SELECT * FROM r WHERE category = '{DROP_CAT}') r LEFT ANTI JOIN t1 ON r.trans_id = t1.id_drop),
    a2 AS (SELECT trans_id, dt, dt AS dt0, acct, amt, nm, cid, ROW_NUMBER() OVER ({grp}) AS rn FROM a_left),
    b2 AS (SELECT trans_id, dt, dt0, acct, amt, nm, cid, ROW_NUMBER() OVER ({grp}) AS rn
           FROM (SELECT trans_id, DATE_SUB(dt, 1) AS dt, dt AS dt0, acct, amt, nm, cid FROM b_left) s),
    t2 AS (SELECT x.trans_id AS id_keep, y.trans_id AS id_drop, x.dt0 AS dt_keep, y.dt0 AS dt_drop, x.acct, x.amt,
                  x.cid AS cid_keep, y.cid AS cid_drop, 2 AS tier
           FROM a2 x JOIN b2 y ON {on})
    SELECT /*+ REPARTITION({FILES['dedup_5c6c']}) */ id_drop AS trans_id_drop, id_keep AS trans_id_keep, tier,
           dt_drop, dt_keep, acct, amt AS amount, cid_drop AS cpty_id_drop, cid_keep AS cpty_id_keep
    FROM (SELECT * FROM t1 UNION ALL SELECT * FROM t2) p
    WHERE dt_drop >= DATE '{lo}'"""

# rows_m: the month's rows after dropping the 6C twins, read from the cached source (src_m) and the staged dedup (dd_m)
_attr_scoped = "".join(f",\n           CASE WHEN s.code IN ({L(codes)}) THEN s.a_{name} END AS {name}"
                       for name, (_, codes) in ATTR_FIELDS.items())
ROWS_M_SQL = f"""
    CREATE OR REPLACE TEMP VIEW rows_m AS
    SELECT s.trans_id, s.txn_date, s.category, s.code, s.amt, s.non_usd, s.cid,
           CASE WHEN s.mp IS NOT NULL THEN 'CUST' ELSE 'CPTY' END AS src_kind, COALESCE(s.mp, s.cid) AS src_id,
           CASE WHEN s.mr IS NOT NULL THEN 'CUST' ELSE 'CPTY' END AS dst_kind, COALESCE(s.mr, s.cid) AS dst_id,
           CASE WHEN s.mp IS NOT NULL AND s.mr IS NOT NULL THEN CAST(NULL AS STRING) ELSE s.cid_src END AS cpty_id_src,
           (s.mp IS NOT NULL AND s.mr IS NOT NULL AND s.mp = s.mr) AS is_self,
           s.cpty_name, s.cpty_type, s.fi{_attr_scoped}
    FROM (
      SELECT * FROM src_m WHERE category <> '{DROP_CAT}'
      UNION ALL
      SELECT x.* FROM (SELECT * FROM src_m WHERE category = '{DROP_CAT}') x
      LEFT ANTI JOIN dd_m d ON x.trans_id = d.trans_id_drop
    ) s"""

_ek = "src_id, src_kind, dst_id, dst_kind, category, cpty_id_src, is_self"
DAILY_SQL = f"""
    SELECT /*+ REPARTITION({FILES['edge_daily']}, src_id) */ * FROM (
      SELECT txn_date, {_ek},
             CAST(SUM(amt) AS DECIMAL(18,2)) AS amount, CAST(COUNT(*) AS INT) AS n_txn,
             CAST(SUM(non_usd) AS INT) AS n_non_usd,
             CAST(MIN(amt) AS DECIMAL(18,2)) AS amt_min, CAST(MAX(amt) AS DECIMAL(18,2)) AS amt_max
      FROM rows_m
      GROUP BY txn_date, {_ek}
    ) d SORT BY src_id, dst_id, txn_date"""

MONTHLY_SQL = f"""
    SELECT /*+ REPARTITION({FILES['edge_monthly']}, src_id) */ * FROM (
      SELECT {_ek},
             CAST(SUM(amount) AS DECIMAL(18,2)) AS amount, CAST(SUM(n_txn) AS INT) AS n_txn,
             CAST(SUM(n_non_usd) AS INT) AS n_non_usd, CAST(COUNT(*) AS INT) AS active_days,
             MIN(txn_date) AS first_dt, MAX(txn_date) AS last_dt,
             CAST(MIN(amt_min) AS DECIMAL(18,2)) AS amt_min, CAST(MAX(amt_max) AS DECIMAL(18,2)) AS amt_max
      FROM d_m
      GROUP BY {_ek}
    ) m SORT BY src_id, dst_id"""

CPTY_MONTH_SQL = f"""
    SELECT /*+ REPARTITION({FILES['cpty_month']}, cpty_id) */ * FROM (
      SELECT cid AS cpty_id, cpty_id_src, category, cpty_type, cpty_name, fi,
             COUNT(*) AS n_txn, CAST(SUM(amt) AS DECIMAL(22,2)) AS amount
      FROM rows_m WHERE cpty_id_src IS NOT NULL
      GROUP BY cid, cpty_id_src, category, cpty_type, cpty_name, fi
    ) c SORT BY cpty_id"""

_attr_cols = ", ".join(ATTR_FIELDS)
CPTY_ATTR_SQL = f"""
    SELECT /*+ REPARTITION({FILES['cpty_attr_month']}, cpty_id) */ * FROM (
      SELECT cid AS cpty_id, cpty_id_src, category, {_attr_cols},
             COUNT(*) AS n_txn, CAST(SUM(amt) AS DECIMAL(22,2)) AS amount
      FROM rows_m WHERE cpty_id_src IS NOT NULL AND code IN ({L(ATTR_CODES)})
      GROUP BY cid, cpty_id_src, category, {_attr_cols}
    ) c SORT BY cpty_id""" if ATTR_FIELDS else None

BUILD_SQL = {"edge_daily": DAILY_SQL, "edge_monthly": MONTHLY_SQL, "cpty_month": CPTY_MONTH_SQL,
             "cpty_attr_month": CPTY_ATTR_SQL}
VIEW_OF = {"dedup_5c6c": "dd_m", "edge_daily": "d_m", "edge_monthly": "mo_m", "cpty_month": "cm_m", "cpty_attr_month": "ca_m"}

# Reconciliation: totals per category from the source side (rows_m) and from every staged output
_attr_case = f"cpty_id_src IS NOT NULL AND code IN ({L(ATTR_CODES)})" if ATTR_FIELDS else "FALSE"
RECON_SQL = {
    "rows": f"""SELECT category, COUNT(*) AS n_rows, CAST(SUM(amt) AS DECIMAL(22,2)) AS amount,
                CAST(SUM(non_usd) AS BIGINT) AS non_usd, CAST(SUM(CASE WHEN is_self THEN 1 ELSE 0 END) AS BIGINT) AS self_rows,
                CAST(SUM(CASE WHEN cpty_id_src IS NOT NULL THEN 1 ELSE 0 END) AS BIGINT) AS ext_rows,
                CAST(SUM(CASE WHEN cpty_id_src IS NOT NULL THEN amt ELSE 0 END) AS DECIMAL(22,2)) AS ext_amount,
                CAST(SUM(CASE WHEN {_attr_case} THEN 1 ELSE 0 END) AS BIGINT) AS attr_rows,
                CAST(SUM(CASE WHEN {_attr_case} THEN amt ELSE 0 END) AS DECIMAL(22,2)) AS attr_amount
                FROM rows_m GROUP BY category""",
    "daily": """SELECT category, CAST(SUM(n_txn) AS BIGINT) AS d_txn, CAST(SUM(amount) AS DECIMAL(22,2)) AS d_amount,
                COUNT(*) AS d_edges, CAST(SUM(n_non_usd) AS BIGINT) AS d_non_usd,
                CAST(SUM(CASE WHEN src_id IS NULL OR dst_id IS NULL THEN 1 ELSE 0 END) AS BIGINT) AS null_id_edges,
                CAST(SUM(CASE WHEN amt_min > amt_max OR amount < amt_min * n_txn OR amount > amt_max * n_txn
                              THEN 1 ELSE 0 END) AS BIGINT) AS d_bad_minmax
                FROM d_m GROUP BY category""",
    "monthly": """SELECT category, CAST(SUM(n_txn) AS BIGINT) AS m_txn, CAST(SUM(amount) AS DECIMAL(22,2)) AS m_amount,
                COUNT(*) AS m_edges, CAST(SUM(active_days) AS BIGINT) AS m_active_days,
                CAST(SUM(CASE WHEN amt_min > amt_max OR first_dt > last_dt THEN 1 ELSE 0 END) AS BIGINT) AS m_bad
                FROM mo_m GROUP BY category""",
    "cpty": """SELECT category, CAST(SUM(n_txn) AS BIGINT) AS c_txn, CAST(SUM(amount) AS DECIMAL(22,2)) AS c_amount
               FROM cm_m GROUP BY category""",
    "attr": """SELECT category, CAST(SUM(n_txn) AS BIGINT) AS a_txn, CAST(SUM(amount) AS DECIMAL(22,2)) AS a_amount
               FROM ca_m GROUP BY category""",
    "dedup": """SELECT COUNT(*) AS n_pairs, CAST(SUM(amount) AS DECIMAL(22,2)) AS amount,
                COUNT(DISTINCT trans_id_drop) AS n_drop, COUNT(DISTINCT trans_id_keep) AS n_keep,
                CAST(SUM(CASE WHEN tier = 2 THEN 1 ELSE 0 END) AS BIGINT) AS tier2 FROM dd_m""",
}

if SHOW_SQL:
    _m = LAST_COMPLETE
    for _name, _q in [("source projection (cached as src_m)", projection(where_months([_m]))), ("5C/6C pairs", dedup_sql(_m)),
                      ("rows_m", ROWS_M_SQL)] + [(k, v) for k, v in BUILD_SQL.items() if v]:
        print(f"-- {_name}{_q}\n")
else:
    print("SQL ready (set SHOW_SQL = True to print it).")

## §1 Plan: which months to build

A month is built when:
- it has never been built (no folder, or a folder without a build marker, e.g. from the previous notebook);
- it was built with other settings (build signature) or its tables come from different builds;
- its source changed since it was built (re-check of the last `RECHECK_LAST` built months: rows, amounts and latest load
  time per category);
- it is listed in `FORCE`.

The completeness gate compares each month with the previous `GATE_WINDOW` months. Earlier months that aren't
profiled yet are profiled here from the source (rows and amounts only, nothing is built).

In [ ]:
# ---- §1 Plan --------------------------------------------------------------------------------------
MONTHS = list(RUN_ONLY) if RUN_ONLY else month_range(RUN_FROM, RUN_TO or LAST_COMPLETE)
assert MONTHS == sorted(MONTHS) and len(set(MONTHS)) == len(MONTHS), "RUN_ONLY must be sorted months without repeats"
assert MONTHS[-1] <= LAST_COMPLETE, f"{MONTHS[-1]} is not complete yet (last complete month {LAST_COMPLETE})"
assert MONTHS[0] >= SRC_FIRST_MONTH, f"the source starts in {SRC_FIRST_MONTH}"

def read_marker(k, m):
    txt = read_text(f"{P[k]}/month={m}/_PKG_BUILD")
    return json.loads(txt) if txt else None

def fingerprint(prof):
    """Source fingerprint of one month: rows, amount and latest load time per category."""
    lines = sorted(f"{r.category}|{int(r.n_rows)}|{Decimal(r.amount):.2f}|{r.max_load_ts}" for r in prof.itertuples())
    ts = [v for v in prof.max_load_ts if v is not None]
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16], (max(ts) if ts else None)

def profile_from_source(ms):
    if not ms: return {}
    pdf = spark.sql(profile_sql(f"({projection(where_months(ms))}) x")).toPandas()
    return {m: pdf[pdf.month == m].reset_index(drop=True) for m in ms}

def save_profiles(prof_by_month, source):
    rows = [p.assign(profiled_at=now_iso(), profile_source=source) for p in prof_by_month.values() if len(p)]
    if not rows: return
    pdf = pd.concat(rows, ignore_index=True)[[c for c, _ in SCHEMA["edge_src_profile"]]]
    spark.createDataFrame(pdf, schema=ddl("edge_src_profile").replace("`", "")).createOrReplaceTempView("_prof_new")
    spark.sql(f"INSERT INTO TABLE {T['edge_src_profile']} SELECT * FROM _prof_new")

def stored_profiles():
    pdf = spark.sql(f"""SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY month, category ORDER BY profiled_at DESC) AS r
                        FROM {T['edge_src_profile']}) x WHERE r = 1""").toPandas()
    return {m: g.drop(columns=["r", "profiled_at", "profile_source"]).reset_index(drop=True) for m, g in pdf.groupby("month")}

PLAN = []
for m in MONTHS:
    mk = {k: read_marker(k, m) for k in KEYS}
    has_dir = any(exists(f"{P[k]}/month={m}") for k in KEYS)
    builds = {v.get("build_id") for v in mk.values() if v}
    row = {"month": m, "action": "build", "reason": "", "build_id": None, "fingerprint": None}
    if m in FORCE:                                     row["reason"] = "forced"
    elif not has_dir:                                  row["reason"] = "new"
    elif any(v is None for v in mk.values()):          row["reason"] = "no build marker (earlier notebook or interrupted)"
    elif len(builds) > 1:                              row["reason"] = "tables from different builds"
    elif any(v["sig"] != SIG for v in mk.values()):    row["reason"] = "settings changed"
    else:
        v = mk[KEYS[0]]
        row.update(action="skip", reason="built", build_id=v["build_id"], fingerprint=v["fingerprint"])
    PLAN.append(row)

# Re-check the source of the last built months
_built = [r for r in PLAN if r["action"] == "skip"]
_recheck = _built[-RECHECK_LAST:] if RECHECK_LAST else []
with timed("re-check source"):
    _rc_prof = profile_from_source([r["month"] for r in _recheck])
RECHECK = []
for r in _recheck:
    fp, ts = fingerprint(_rc_prof[r["month"]])
    changed = fp != r["fingerprint"]
    RECHECK.append({"month": r["month"], "built_fingerprint": r["fingerprint"], "source_now": fp,
                    "latest_load_ts": ts, "changed": changed})
    if changed:
        r.update(action="build", reason=f"source changed since build {r['build_id']}")
save_profiles(_rc_prof, "recheck")

TODO = [r["month"] for r in PLAN if r["action"] == "build"]
if MAX_MONTHS_PER_RUN:
    TODO = TODO[:MAX_MONTHS_PER_RUN]

# Gate history: profiles of the GATE_WINDOW months before each month to build
PROFILES = stored_profiles()
PROFILES.update({m: p for m, p in _rc_prof.items()})
_need = sorted({pm for m in TODO for pm in prev_months(m, GATE_WINDOW)} - set(TODO) - set(PROFILES))
_need = [pm for pm in _need if pm >= SRC_FIRST_MONTH]
with timed("gate history"):
    _hist = profile_from_source(_need)
save_profiles(_hist, "gate_history")
PROFILES.update(_hist)

_plan = pd.DataFrame(PLAN)
_summary = _plan.assign(n=1).groupby(["action", "reason"], as_index=False).n.sum()
out(sec(f"Plan for {MONTHS[0]} → {MONTHS[-1]} ({len(MONTHS)} months): {len(TODO)} to build in this run", tbl(_summary)),
    sec("Months to build", tbl(_plan[_plan.month.isin(TODO)][["month", "reason"]])) if TODO else para("Nothing to build."),
    sec(f"Source re-check of the last {len(RECHECK)} built months", tbl(pd.DataFrame(RECHECK), flag="changed"))
        if RECHECK else para("No built months to re-check."),
    para(f"Gate history profiled from the source: {', '.join(_need) if _need else 'none needed'}."))

## §2 Build

For each month: read and cache, gate, dedup, stage, reconcile, publish. One line is printed per month.
- **Gate STOP:** the run stops before writing that month. Check the source; if the month is really complete, add it to
  `ACCEPT_MONTHS` and run again.
- **Reconciliation mismatch:** the run stops and nothing is published for that month; the previous version stays.

In [ ]:
# ---- §2 Build -------------------------------------------------------------------------------------
for k in KEYS:                                     # leftovers of an interrupted or failed run
    delete(f"{P[k]}/_staging"); delete(f"{P[k]}/_trash")

RAILS = ["ACH", "WIRE", "RTP", "CHECK", "PCARD"]
def by_rail(prof):
    p = prof.assign(rail=prof.category.map(rail_of), amt_f=prof.amount.map(float))
    return p.groupby("rail").agg(n_rows=("n_rows", "sum"), amount=("amt_f", "sum"), max_dt=("max_dt", "max"))

def gate(m, prof):
    cur, last = by_rail(prof), dt.date.fromisoformat(month_bounds(m)[1])
    hist_m = [pm for pm in prev_months(m, GATE_WINDOW) if pm in PROFILES and len(PROFILES[pm])]
    hist = pd.concat([by_rail(PROFILES[pm]).assign(month=pm) for pm in hist_m]) if hist_m else pd.DataFrame()
    rank = {"OK": 0, "WARN": 1, "STOP": 2}
    rows = []
    for rail in [r for r in RAILS if r in cur.index or (len(hist) and r in hist.index)]:
        h = hist.loc[[rail]] if len(hist) and rail in hist.index else pd.DataFrame()
        med_rows = float(h.n_rows.median()) if len(h) else np.nan
        med_usd = float(h.amount.median()) if len(h) else np.nan
        flags = []
        if rail not in cur.index:
            flags.append(("STOP", "no rows this month")); c = None
        else:
            c = cur.loc[rail]
            behind = (last - c.max_dt).days
            if behind > GATE_TAIL_DAYS.get(rail, 4):
                flags.append(("STOP", f"last date {c.max_dt} ({behind} days before month end)"))
            if len(h):
                rr = c.n_rows / med_rows - 1
                if rr < -GATE_ROWS_LOW:    flags.append(("STOP", f"rows {rr:+.0%} vs median"))
                elif rr > GATE_ROWS_HIGH:  flags.append(("WARN", f"rows {rr:+.0%} vs median"))
                if med_usd:
                    ru = c.amount / med_usd - 1
                    if abs(ru) > GATE_USD: flags.append(("WARN", f"dollars {ru:+.0%} vs median"))
        level = max([f[0] for f in flags], key=rank.get, default="OK")
        rows.append({"month": m, "rail": rail, "n_rows": int(c.n_rows) if c is not None else 0,
                     "rows_vs_median": (c.n_rows / med_rows - 1) if c is not None and len(h) else np.nan,
                     "dollars": c.amount if c is not None else 0.0,
                     "dollars_vs_median": (c.amount / med_usd - 1) if c is not None and len(h) and med_usd else np.nan,
                     "last_date": str(c.max_dt) if c is not None else None, "flag": level,
                     "note": "; ".join(f[1] for f in flags)})
    det = pd.DataFrame(rows)
    verdict = max(det.flag, key=rank.get) if len(det) else "STOP"
    notes = "; ".join(f"{r.rail}: {r.note}" for r in det.itertuples() if r.note)
    if not hist_m:
        notes = (notes + "; " if notes else "") + "no earlier months to compare (tail check only)"
    if verdict == "STOP" and m in ACCEPT_MONTHS:
        verdict = "ACCEPTED"
    return {"month": m, "verdict": verdict, "notes": notes, "history": ", ".join(hist_m), "detail": det}

def stage(key, query, m, build_id):
    """Write one output of month m to its staging folder and expose it as a temp view for the next steps."""
    path = f"{P[key]}/_staging/month={m}/{build_id}"
    if query.lstrip().upper().startswith("WITH"):  # Spark SQL wants CTEs before INSERT; wrap them instead
        query = f"SELECT * FROM ({query}) q"
    spark.sql(f"INSERT OVERWRITE DIRECTORY '{path}' USING parquet OPTIONS ('compression' '{CODEC}') {query}")
    spark.read.schema(ddl(key).replace("`", "")).parquet(path).createOrReplaceTempView(VIEW_OF[key])
    return path

def staged_schema_ok(key, path):
    if not [n for n in list_names(path) if n.endswith(".parquet")]:
        return True
    have = [(f.name.lower(), f.dataType.simpleString()) for f in spark.read.parquet(path).schema]
    return have == SCHEMA[key]

def _d(v):
    """Exact number for comparisons: Decimal amounts stay Decimal, counts become Decimal integers, missing = 0."""
    if v is None: return Decimal(0)
    if isinstance(v, Decimal): return v
    if isinstance(v, (float, np.floating)):
        return Decimal(0) if np.isnan(v) else Decimal(str(v))
    return Decimal(int(v))

def reconcile(m, prof):
    q = {k: spark.sql(v).toPandas() for k, v in RECON_SQL.items() if k != "attr" or ATTR_FIELDS}
    dd = q.pop("dedup").iloc[0]
    df = prof[["category", "n_rows", "amount"]].rename(columns={"n_rows": "raw_rows", "amount": "raw_amount"})
    for k in ("rows", "daily", "monthly", "cpty", "attr"):
        if k in q:
            df = df.merge(q[k], on="category", how="outer")
    for c in df.columns:
        if c != "category":
            df[c] = df[c].map(_d)
    df["dropped"] = [_d(dd.n_pairs) if c == DROP_CAT else Decimal(0) for c in df.category]
    df["dropped_amount"] = [_d(dd.amount) if c == DROP_CAT else Decimal(0) for c in df.category]
    checks = {
        "source − 6C duplicates = rows":   (df.raw_rows - df.dropped == df.n_rows) & (df.raw_amount - df.dropped_amount == df.amount),
        "rows = daily edges":              (df.n_rows == df.d_txn) & (df.amount == df.d_amount) & (df.non_usd == df.d_non_usd),
        "daily = monthly edges":           (df.d_txn == df.m_txn) & (df.d_amount == df.m_amount) & (df.d_edges == df.m_active_days),
        "external rows = cpty_month":      (df.ext_rows == df.c_txn) & (df.ext_amount == df.c_amount),
        "min/max consistent":              (df.d_bad_minmax == 0) & (df.m_bad == 0),
    }
    if ATTR_FIELDS:
        checks["attribute rows = cpty_attr_month"] = (df.attr_rows == df.a_txn) & (df.attr_amount == df.a_amount)
    pairs_ok = int(dd.n_pairs) == int(dd.n_drop) == int(dd.n_keep)
    df["ok"] = np.logical_and.reduce(list(checks.values()))
    summary = {name: bool(s.all()) for name, s in checks.items()}
    summary["dedup pairs one-to-one"] = pairs_ok
    return df, summary, dd

def log_run(**kw):
    row = {c: kw.get(c) for c, _ in SCHEMA["edge_runlog"]}
    pdf = pd.DataFrame([row])
    for c, t in SCHEMA["edge_runlog"]:
        if t.startswith("decimal"):
            pdf[c] = pdf[c].map(lambda v: None if v is None else Decimal(v))
    spark.createDataFrame(pdf, schema=ddl("edge_runlog").replace("`", "")).createOrReplaceTempView("_log_new")
    spark.sql(f"INSERT INTO TABLE {T['edge_runlog']} SELECT * FROM _log_new")

def publish(m, build_id, staged, marker):
    for k in KEYS:                                 # the marker travels with the data
        write_text(f"{staged[k]}/_PKG_BUILD", json.dumps(dict(marker, table=T[k])))
    for k in KEYS:
        final = f"{P[k]}/month={m}"
        mkdirs(P[k])
        if exists(final):
            mkdirs(f"{P[k]}/_trash"); rename(final, f"{P[k]}/_trash/month={m}.{build_id}")
        rename(staged[k], final)
        spark.sql(f"ALTER TABLE {T[k]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{final}'")
    for k in KEYS:
        delete(f"{P[k]}/_trash"); delete(f"{P[k]}/_staging"); spark.sql(f"REFRESH TABLE {T[k]}")

FAIL_COLS = ["category", "raw_rows", "dropped", "n_rows", "d_txn", "m_txn", "ext_rows", "c_txn", "attr_rows", "a_txn",
             "amount", "d_amount", "m_amount", "ok"]
GATE_LOG, QA_LOG, BUILT = [], [], []
print(f"{len(TODO)} month(s) to build" + (f": {TODO[0]} → {TODO[-1]}" if TODO else ""))
for m in TODO:
    build_id = f"{dt.datetime.now():%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
    started, t0, logged = now_iso(), time.time(), False
    g, fp, ts = None, None, None
    try:
        with timed("read + cache source", m):
            spark.sql("UNCACHE TABLE IF EXISTS src_m"); spark.sql("DROP VIEW IF EXISTS src_m")
            spark.sql(f"CACHE TABLE src_m AS {projection(where_months([m]))}")
        with timed("profile + gate", m):
            prof = spark.sql(profile_sql("src_m")).toPandas()
            save_profiles({m: prof}, "build")
            PROFILES[m] = prof
            fp, ts = fingerprint(prof)
            g = gate(m, prof); GATE_LOG.append(g)
        if g["verdict"] == "STOP":
            log_run(build_id=build_id, month=m, status="stopped", reason="completeness gate", gate=g["verdict"],
                    gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts, src_rows=int(prof.n_rows.sum()),
                    minutes=(time.time() - t0) / 60, started_at=started, finished_at=now_iso())
            logged = True
            out(sec(f"{m}: the completeness gate stopped the run", tbl(g["detail"], pct=("rows_vs_median", "dollars_vs_median"),
                    flag="flag")), para("Nothing was written for this month. If the source is complete, add the month "
                                        "to ACCEPT_MONTHS and run again."))
            raise AssertionError(f"{m}: completeness gate STOP: {g['notes']}")

        staged = {}
        with timed("5C/6C pairs", m):
            staged["dedup_5c6c"] = stage("dedup_5c6c", dedup_sql(m), m, build_id)
            spark.sql(ROWS_M_SQL)
        for k, label in [("edge_daily", "daily edges"), ("edge_monthly", "monthly edges"),
                         ("cpty_month", "counterparty-month"), ("cpty_attr_month", "counterparty attributes")]:
            if k in KEYS:
                with timed(label, m):
                    staged[k] = stage(k, BUILD_SQL[k], m, build_id)
        with timed("reconciliation", m):
            qa, summary, dd = reconcile(m, prof)
            bad_schema = [k for k in KEYS if not staged_schema_ok(k, staged[k])]
        QA_LOG.append({"month": m, "qa": qa, "summary": summary, "dedup": dd, "gate": g["verdict"]})
        if not all(summary.values()) or bad_schema:
            log_run(build_id=build_id, month=m, status="failed", reason="reconciliation: " + ", ".join(
                        [k for k, v in summary.items() if not v] + [f"schema {k}" for k in bad_schema]),
                    gate=g["verdict"], gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts,
                    src_rows=int(prof.n_rows.sum()), minutes=(time.time() - t0) / 60, started_at=started, finished_at=now_iso())
            logged = True
            out(sec(f"{m}: RECONCILIATION FAILED (nothing published)", tbl(pd.DataFrame(
                    [{"check": k, "ok": v} for k, v in summary.items()] + [{"check": f"schema of {k}", "ok": False} for k in bad_schema]),
                    flag="ok")), sec("Categories that failed", tbl(qa[~qa.ok][[c for c in FAIL_COLS if c in qa.columns]], flag="ok")))
            raise AssertionError(f"{m}: reconciliation failed, nothing published")

        n_rows, amount = int(sum(qa.n_rows)), sum(qa.amount)
        d_edges, m_edges = int(sum(qa.d_edges)), int(sum(qa.m_edges))
        with timed("publish", m):
            publish(m, build_id, staged, {"month": m, "build_id": build_id, "sig": SIG, "fingerprint": fp,
                                          "max_load_ts": ts, "built_at": now_iso(), "edge_rows": n_rows})
        mins = (time.time() - t0) / 60
        log_run(build_id=build_id, month=m, status="published", reason=next(r["reason"] for r in PLAN if r["month"] == m),
                gate=g["verdict"], gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts,
                src_rows=int(prof.n_rows.sum()), dropped_6c=int(dd.n_pairs), edge_rows=n_rows, amount=amount,
                daily_edges=d_edges, monthly_edges=m_edges, minutes=mins, started_at=started, finished_at=now_iso())
        logged = True
        BUILT.append(m)
        print(f"{m}: published in {mins:,.1f} min | gate {g['verdict']} | {n_rows:,} rows, "
              f"{int(dd.n_pairs):,} 6C duplicates dropped | {d_edges:,} daily, {m_edges:,} monthly edges | build {build_id}")
    except Exception as e:
        if not logged:
            log_run(build_id=build_id, month=m, status="failed", reason=f"{type(e).__name__}: {str(e)[:400]}",
                    gate=g["verdict"] if g else None, sig=SIG, fingerprint=fp, max_load_ts=ts,
                    minutes=(time.time() - t0) / 60, started_at=started, finished_at=now_iso())
        raise
    finally:
        spark.sql("UNCACHE TABLE IF EXISTS src_m")

for k in KEYS + LOG_KEYS:
    spark.sql(f"REFRESH TABLE {T[k]}")
print(f"done: {len(BUILT)} month(s) published" + (f" ({BUILT[0]} → {BUILT[-1]})" if BUILT else ""))

## §3 Reconciliation and QA (months built in this run)

In [ ]:
# ---- §3 Reconciliation and QA ----------------------------------------------------------------------
if QA_LOG:
    per_m = pd.DataFrame([{
        "month": r["month"], "source_rows": int(sum(r["qa"].raw_rows)), "dropped_6c": int(r["dedup"].n_pairs),
        "tier2_pairs": int(r["dedup"].tier2), "edge_rows": int(sum(r["qa"].n_rows)), "amount": float(sum(r["qa"].amount)),
        "daily_edges": int(sum(r["qa"].d_edges)), "monthly_edges": int(sum(r["qa"].m_edges)),
        "null_id_edges": int(sum(r["qa"].null_id_edges)), "gate": r["gate"],
        "reconciled": all(r["summary"].values())} for r in QA_LOG])
    last = QA_LOG[-1]
    d = last["qa"].copy()
    d.insert(1, "code", d.category.map(code_of)); d.insert(2, "rail", d.category.map(rail_of))
    n = d.n_rows.map(float)
    d["daily_per_row"] = d.d_edges.map(float) / n
    d["monthly_per_row"] = d.m_edges.map(float) / n
    d["self_share"] = d.self_rows.map(float) / n
    d["non_usd_share"] = d.non_usd.map(float) / n
    cols = ["code", "rail", "raw_rows", "dropped", "n_rows", "amount", "d_edges", "daily_per_row", "m_edges",
            "monthly_per_row", "ext_rows", "attr_rows", "null_id_edges", "self_share", "non_usd_share", "ok"]
    checks = pd.DataFrame([{"check": k, "ok": v} for k, v in last["summary"].items()])
    out(sec("Per month: rows, duplicates, edges, reconciliation", tbl(per_m, flag="reconciled")),
        sec(f"Checks for {last['month']} (every month passed the same checks before publishing)", tbl(checks, flag="ok")),
        sec(f"Per category, {last['month']}", tbl(d[cols].sort_values("code"),
            pct=("daily_per_row", "monthly_per_row", "self_share", "non_usd_share"), flag="ok")))
else:
    out(para("Nothing was built in this run."))

## §4 Completeness gate (months evaluated in this run)

One row per month; the rail detail follows for months with a warning. A warning doesn't stop the build: it points at
months worth a look (for example the 2026-07/08 dollar jump).

In [ ]:
# ---- §4 Completeness gate ----------------------------------------------------------------------------
if GATE_LOG:
    s = pd.DataFrame([{"month": g["month"], "verdict": g["verdict"], "compared_with": g["history"] or "—",
                       "notes": g["notes"] or "—"} for g in GATE_LOG])
    flagged = [g["detail"] for g in GATE_LOG if g["verdict"] != "OK"]
    parts = [sec("Gate per month", tbl(s, flag="verdict"))]
    if flagged:
        fd = pd.concat(flagged, ignore_index=True)
        parts.append(sec("Rail detail for flagged months", tbl(fd[fd.flag != "OK"], pct=("rows_vs_median", "dollars_vs_median"),
                                                                 flag="flag")))
    out(*parts)
else:
    out(para("No month was evaluated in this run."))

## §5 Counterparty attribute coverage

For the latest month built, the share of rows and dollars where each attribute field is filled, in every kept
category. Use it to decide the fields and categories in `ATTR_FIELDS`: a field that is essentially empty can be
dropped, and a category where a field is well filled can be added. Changing `ATTR_FIELDS` rebuilds every month.

In [ ]:
# ---- §5 Attribute coverage ---------------------------------------------------------------------------
_pm = BUILT[-1] if BUILT else next((r["month"] for r in reversed(PLAN) if r["action"] == "skip"), None)
if PROFILE_ATTRS and ATTR_FIELDS and _pm:
    _fill = ",\n".join(
        f"CAST(SUM(CASE WHEN a_{k} IS NOT NULL THEN 1 ELSE 0 END) AS DOUBLE) / COUNT(*) AS r_{k}, "
        f"CAST(SUM(CASE WHEN a_{k} IS NOT NULL THEN amt ELSE 0 END) AS DOUBLE) / CAST(SUM(amt) AS DOUBLE) AS u_{k}, "
        f"APPROX_COUNT_DISTINCT(a_{k}) AS n_{k}" for k in ATTR_FIELDS)
    cov = spark.sql(f"""SELECT category, COUNT(*) AS n_rows, CAST(SUM(amt) AS DOUBLE) AS amount, {_fill}
                        FROM ({projection(where_months([_pm]))}) x GROUP BY category""").toPandas()
    cov["code"] = cov.category.map(code_of)
    cov = cov.sort_values("code").reset_index(drop=True)
    _cell = lambda r, u: "—" if not r else f"{r:.0%} · {u:.0%}"
    wide = cov[["code", "n_rows", "amount"]].copy()
    for k in ATTR_FIELDS:
        wide[k] = [_cell(r, u) for r, u in zip(cov[f"r_{k}"].fillna(0), cov[f"u_{k}"].fillna(0))]
    fields = pd.DataFrame([{"field": k, "source_column": v[0], "kept_for": ", ".join(v[1]),
                            "distinct_values_there": int(cov[cov.code.isin(v[1])][f"n_{k}"].sum())}
                           for k, v in ATTR_FIELDS.items()])
    rows_ratio = spark.sql(f"""SELECT c.category, c.cm_rows, a.attr_rows FROM
        (SELECT category, COUNT(*) AS cm_rows FROM {T['cpty_month']} WHERE month = '{_pm}' GROUP BY category) c
        JOIN (SELECT category, COUNT(*) AS attr_rows FROM {T['cpty_attr_month']} WHERE month = '{_pm}' GROUP BY category) a
        ON c.category = a.category""").toPandas()
    rows_ratio["attr_rows_per_cpty_month_row"] = rows_ratio.attr_rows / rows_ratio.cm_rows
    out(sec(f"Fields kept ({_pm})", tbl(fields)),
        sec("Filled share in every kept category: rows · dollars (— = never filled)", tbl(wide)),
        sec("Size of pkg_cpty_attr_month vs pkg_cpty_month (rows split by attribute values)",
            tbl(rows_ratio, dec=("attr_rows_per_cpty_month_row",))))
else:
    out(para("Attribute coverage skipped (PROFILE_ATTRS is False, ATTR_FIELDS is empty, or no month is built)."))

## §6 Status, storage and time

In [ ]:
# ---- §6 Status, storage and time ---------------------------------------------------------------------
status = []
for m in MONTHS:
    mk = {k: read_marker(k, m) for k in KEYS}
    ok = all(v and v["sig"] == SIG for v in mk.values()) and len({v["build_id"] for v in mk.values() if v}) == 1
    v = mk[KEYS[0]] or {}
    status.append({"month": m, "built": ok, "build_id": v.get("build_id"), "built_at": v.get("built_at"),
                   "edge_rows": v.get("edge_rows")})
st = pd.DataFrame(status)
built_m = list(st[st.built].month)
size = []
for k in KEYS:
    s = [size_of(f"{P[k]}/month={m}") for m in built_m]
    b = sum(x[0] for x in s)
    size.append({"table": T[k], "months": len(built_m), "GB_total": b / GB, "GB_per_month": b / max(len(built_m), 1) / GB,
                 f"GB_on_HDFS_x{REPL}": b * REPL / GB, "files_per_month": sum(x[1] for x in s) / max(len(built_m), 1)})
S = pd.DataFrame(size)
S = pd.concat([S, pd.DataFrame([{"table": "TOTAL", "months": len(built_m),
              **{c: S[c].sum() for c in S.columns if c not in ("table", "months")}}])], ignore_index=True)
parts = [sec(f"Months {MONTHS[0]} → {MONTHS[-1]}: {len(built_m)} of {len(MONTHS)} built with the current settings",
             tbl(st if len(st) <= 40 else st[~st.built | st.month.isin(BUILT)], flag="built")),
         sec("Storage", tbl(S, dec=tuple(c for c in S.columns if c.startswith("GB") or c == "files_per_month")))]
try:
    q = fs_for(DB_LOC).getQuotaUsage(JPath(DB_LOC))
    if q.getSpaceQuota() >= 0:
        parts.append(para(f"Space quota at {DB_LOC}: {(q.getSpaceQuota() - q.getSpaceConsumed()) / GB / 1000:,.2f} TB "
                          "remaining (replicated)."))
except Exception as e:
    parts.append(para(f"Quota lookup skipped: {str(e)[:100]}"))
if TIMINGS:
    TM = pd.DataFrame(TIMINGS)
    per_stage = (TM[TM.month.notna()].groupby("stage", sort=False).seconds.mean() / 60).reset_index(name="minutes_per_month")
    per_month = TM[TM.month.notna()].groupby("month").seconds.sum() / 60
    remaining = len(MONTHS) - len(built_m)
    parts.append(sec("Time per stage (mean over months built in this run)", tbl(per_stage, dec=("minutes_per_month",))))
    parts.append(para(f"Months built in this run: {len(per_month)}, {per_month.mean():,.1f} min each on average. "
                      f"Re-check {TM[TM.stage == 're-check source'].seconds.sum() / 60:,.1f} min, gate history "
                      f"{TM[TM.stage == 'gate history'].seconds.sum() / 60:,.1f} min. Months still to build: {remaining}"
                      + (f" (about {remaining * per_month.mean() / 60:,.1f} h)." if remaining and len(per_month) else ".")))
out(*parts)

## §7 Impala

The tables are standard external parquet tables. Run the printed statements in Impala: `INVALIDATE METADATA` for
tables created or re-defined in this run, `REFRESH` for the others.

In [ ]:
# ---- §7 Impala -----------------------------------------------------------------------------------------
for k in KEYS + LOG_KEYS:
    if TABLE_ACTIONS[k] != "ok":
        print(f"INVALIDATE METADATA {T[k]};   -- {TABLE_ACTIONS[k]}")
    else:
        print(f"REFRESH {T[k]};")
print(f"\n-- optional, faster joins:\nCOMPUTE INCREMENTAL STATS {T['edge_monthly']};")